# Task C: Catheter Length Estimation

**Methodology**:
1. Skeletonize the Task B catheter mask to get the visible centerline
2. Compute the arc length of the visible centerline (in mm)
3. Find where the visible catheter meets the Task A artifact mask
4. Extrapolate the catheter trajectory through the artifact using the local direction
5. Hidden length = distance traveled through the artifact along the extrapolated path
6. Total length = visible + hidden

*Note: depends on task A and task B

**Output:** `pred_dir/taskC.csv`

In [ ]:
%%capture
!pip install nibabel scikit-image matplotlib

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print('Colab:', IN_COLAB)

In [ ]:
import csv
import json
from collections import deque
from pathlib import Path

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize
from tqdm.auto import tqdm

print('Imports OK.')

In [ ]:
if IN_COLAB:
    DATA_ROOT = Path('/content/drive/MyDrive/Project 2')
else:
    DATA_ROOT = Path('/Volumes/LUCY DISK/mia/Project 2')

GT_DIR   = DATA_ROOT / 'gt_dir'
PRED_DIR = DATA_ROOT / 'pred_dir'

SUBJECTS = [f'subj{i:03d}' for i in range(1, 31)]

# Number of skeleton voxels near the artifact endpoint used to estimate
# the local catheter direction for extrapolation.
N_FIT = 15

print('GT dir exists:', GT_DIR.exists())
print('Pred dir exists:', PRED_DIR.exists())

In [ ]:
# ── Geometry helpers ───────────────────────────────────────────────────────

def load_spacing(json_path: Path):
    """Return (sx, sy, sz) voxel spacing in mm from subject JSON."""
    with open(json_path) as f:
        meta = json.load(f)
    zyx = meta.get('spacing_zyx_mm', [1.0, 1.0, 1.0])
    return (float(zyx[2]), float(zyx[1]), float(zyx[0]))


def skeleton_length_mm(skel: np.ndarray, spacing=(1.0, 1.0, 1.0)) -> float:
    """
    Arc length of a 3D binary skeleton in mm.
    Sums Euclidean edge weights over all 26-connected adjacencies,
    then halves (each edge is counted from both endpoints).
    """
    pts = np.array(np.where(skel)).T
    if len(pts) < 2:
        return 0.0
    pt_set = set(map(tuple, pts))
    sx, sy, sz = spacing
    total = 0.0
    for x, y, z in map(tuple, pts):
        for dx in (-1, 0, 1):
            for dy in (-1, 0, 1):
                for dz in (-1, 0, 1):
                    if dx == dy == dz == 0:
                        continue
                    if (x + dx, y + dy, z + dz) in pt_set:
                        total += np.sqrt((dx * sx) ** 2 + (dy * sy) ** 2 + (dz * sz) ** 2)
    return total / 2.0


def skeleton_endpoints(skel_pts: np.ndarray):
    """Return skeleton voxels with exactly one 26-connected neighbour."""
    pt_set = set(map(tuple, skel_pts))
    endpoints = []
    for pt in map(tuple, skel_pts):
        n_nb = sum(
            1
            for dx in (-1, 0, 1)
            for dy in (-1, 0, 1)
            for dz in (-1, 0, 1)
            if (dx, dy, dz) != (0, 0, 0)
            and (pt[0] + dx, pt[1] + dy, pt[2] + dz) in pt_set
        )
        if n_nb == 1:
            endpoints.append(pt)
    return endpoints


def local_direction(skel_pts: np.ndarray, endpoint: tuple, n: int = 15) -> np.ndarray:
    """
    BFS from endpoint to collect the first n skeleton voxels,
    then use PCA (SVD) to find the dominant direction of that segment.
    The direction is oriented to point away from the endpoint (outward).
    """
    pt_set = set(map(tuple, skel_pts))
    visited = {endpoint}
    order = [endpoint]
    queue = deque([endpoint])
    while queue and len(order) < n:
        curr = queue.popleft()
        for dx in (-1, 0, 1):
            for dy in (-1, 0, 1):
                for dz in (-1, 0, 1):
                    if (dx, dy, dz) == (0, 0, 0):
                        continue
                    nb = (curr[0] + dx, curr[1] + dy, curr[2] + dz)
                    if nb in pt_set and nb not in visited:
                        visited.add(nb)
                        order.append(nb)
                        queue.append(nb)
    pts = np.array(order, dtype=float)
    if len(pts) < 2:
        return np.array([1.0, 0.0, 0.0])
    _, _, vt = np.linalg.svd(pts - pts.mean(axis=0))
    direction = vt[0]
    # Orient so it points away from the bulk of the skeleton (outward from endpoint)
    outward = np.array(order[-1], dtype=float) - np.array(order[0], dtype=float)
    if np.dot(direction, outward) < 0:
        direction = -direction
    return direction


def trace_through_artifact(
    start_vox: tuple,
    direction: np.ndarray,
    artifact_mask: np.ndarray,
    step: float = 0.5,
    max_steps: int = 300,
) -> float:
    """
    Step from start_vox in direction (voxel coords, isotropic 1mm spacing).
    Returns the distance (mm) traveled inside the artifact mask.
    Stops after exiting the artifact or hitting the volume boundary.
    """
    shape = np.array(artifact_mask.shape)
    pos   = np.array(start_vox, dtype=float)
    direction = direction / (np.linalg.norm(direction) + 1e-8)
    hidden = 0.0
    entered = False

    for _ in range(max_steps):
        pos += direction * step
        vox = np.round(pos).astype(int)
        if np.any(vox < 0) or np.any(vox >= shape):
            break
        in_artifact = bool(artifact_mask[vox[0], vox[1], vox[2]])
        if in_artifact:
            entered = True
            hidden += step
        elif entered:
            break  # exited the artifact — stop counting

    return hidden


print('Helpers defined.')

In [ ]:
def estimate_catheter_length(taskB_mask: np.ndarray,
                             taskA_mask: np.ndarray,
                             spacing=(1.0, 1.0, 1.0)) -> float:
    """
    Estimate total catheter length (mm) = visible centerline + hidden through artifact.

    Fallback when Task B predicts nothing: returns the artifact diameter as a
    rough lower bound (the catheter must at least cross the artifact).
    """
    artifact_pts = np.array(np.where(taskA_mask > 0)).T

    # ── Fallback: no catheter predicted ────────────────────────────────────
    if taskB_mask.sum() == 0:
        if len(artifact_pts) == 0:
            return 0.0
        # Artifact bounding-box diagonal as a rough minimum
        bbox_diag = np.linalg.norm(
            (artifact_pts.max(axis=0) - artifact_pts.min(axis=0)) * np.array(spacing)
        )
        return float(bbox_diag)

    # ── Step 1: skeletonise visible catheter ───────────────────────────────
    skel = skeletonize(taskB_mask.astype(bool))
    visible_length = skeleton_length_mm(skel, spacing)

    skel_pts = np.array(np.where(skel)).T
    if len(skel_pts) == 0 or len(artifact_pts) == 0:
        return visible_length

    # ── Step 2: find skeleton endpoint nearest the artifact ─────────────────
    endpoints = skeleton_endpoints(skel_pts)
    if not endpoints:
        endpoints = [tuple(skel_pts[0])]

    artifact_centroid = artifact_pts.mean(axis=0)
    nearest_ep = min(endpoints,
                     key=lambda ep: np.linalg.norm(np.array(ep) - artifact_centroid))

    # ── Step 3: local direction at that endpoint ────────────────────────────
    direction = local_direction(skel_pts, nearest_ep, n=N_FIT)

    # Ensure direction points toward the artifact centroid
    to_artifact = artifact_centroid - np.array(nearest_ep)
    if np.dot(direction, to_artifact) < 0:
        direction = -direction

    # ── Step 4: trace through artifact ─────────────────────────────────────
    hidden_length = trace_through_artifact(
        nearest_ep, direction, taskA_mask > 0
    )

    return visible_length + hidden_length


print('estimate_catheter_length defined.')

In [ ]:
# Run over all subjects and write taskC.csv

results = []

for subj in tqdm(SUBJECTS):
    taskB_path = PRED_DIR / f'{subj}_taskB.nii.gz'
    taskA_path = PRED_DIR / f'{subj}_taskA.nii.gz'

    if not taskB_path.exists():
        print(f'[SKIP] {subj}: no Task B prediction')
        continue

    taskB_mask = nib.load(str(taskB_path)).get_fdata()
    taskA_mask = nib.load(str(taskA_path)).get_fdata() if taskA_path.exists() else np.zeros_like(taskB_mask)

    # Load spacing from JSON if available
    json_path = GT_DIR / subj / f'{subj}.json'
    spacing = load_spacing(json_path) if json_path.exists() else (1.0, 1.0, 1.0)

    length_mm = estimate_catheter_length(taskB_mask, taskA_mask, spacing)
    results.append({'subject': subj, 'length': round(length_mm, 4)})

# Save taskC.csv
taskC_path = PRED_DIR / 'taskC.csv'
with open(taskC_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['subject', 'length'])
    writer.writeheader()
    writer.writerows(results)

print(f'\nSaved {len(results)} entries to {taskC_path}')
for r in results:
    print(f"  {r['subject']}: {r['length']:.2f} mm")

In [ ]:
# Evaluate against GT lengths from JSON files (where available)

def relative_absolute_error(pred, gt):
    if gt == 0:
        return 0.0 if pred == 0 else float('inf')
    return abs(pred - gt) / gt


errors, gt_lengths, pred_lengths = [], [], []

for r in results:
    subj = r['subject']
    json_path = GT_DIR / subj / f'{subj}.json'
    if not json_path.exists():
        continue
    with open(json_path) as f:
        gt_len = float(json.load(f).get('catheter_length_mm', 0))
    pred_len = r['length']
    err = relative_absolute_error(pred_len, gt_len)
    errors.append(err)
    gt_lengths.append(gt_len)
    pred_lengths.append(pred_len)
    print(f"{subj}: pred={pred_len:.1f} mm  gt={gt_len:.1f} mm  rel_err={err:.3f}")

if errors:
    print(f'\nMean relative absolute error: {np.mean(errors):.4f}')

In [ ]:
# Scatter plot: predicted vs GT length
if gt_lengths:
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(gt_lengths, pred_lengths, alpha=0.7)
    lim = max(max(gt_lengths), max(pred_lengths)) * 1.05
    ax.plot([0, lim], [0, lim], 'r--', label='perfect prediction')
    ax.set(xlabel='GT length (mm)', ylabel='Predicted length (mm)',
           title='Task C: Predicted vs GT catheter length')
    ax.legend()
    plt.tight_layout()
    plt.show()